# Lamoda · Проверка верхней части рейтинга

Запуск из корня репозитория с архивом соревнования или переменной `LAMODA_DATA`. В окружение предыдущей модели добавьте CatBoost 1.2.10 и XGBoost CPU 3.4.1. Ноутбук формирует апрельский CSV и ZIP. Кандидата выбираем по февралю, проверяем в марте. Если он уступает контролю хотя бы раз, итоговый файл сохраняет прогноз квартальной модели.

In [1]:
%run ./lamoda_context.ipynb

               size     mean
scoring_date                
2022-01-01    25000  0.34764
2022-02-01    25000  0.33368
2022-03-01    25000  0.33924


Матрицы признаков: (75000, 29) (25000, 29)


699 признаков


Переходы и все признаки: (75000, 733) (75000, 861)


date                  2022-02-01  2022-03-01     mean
model                                                
combined_regional        0.76002     0.77011  0.76506
full_regional            0.75932     0.76985  0.76458
transitions_regional     0.75930     0.76812  0.76371
transitions              0.75465     0.76282  0.75874
regional                 0.74834     0.76359  0.75597
Выбрано по средней AP@20%: combined_regional
Проверены 25,000 строк и порядок клиентов по образцу Kaggle.


Новых признаков 459


      date    model     ap20  buyers_in_top_5000
2022-02-01 previous 0.760016                4114
2022-02-01   motifs 0.774831                4159
2022-03-01 previous 0.770113                4142
2022-03-01   motifs 0.782566                4169


25000 клиентов в submission_motifs.csv


      date    model     ap20  buyers  changed  added_buyers  dropped_buyers  false_7_15
2022-02-01   motifs 0.774831    4159        0             0               0         333
2022-02-01  context 0.776896    4166      135            97              90         338
2022-02-01 quarters 0.777011    4166      136            94              87         338
2022-03-01   motifs 0.782566    4169        0             0               0         321
2022-03-01  context 0.784056    4170      108            65              64         322
2022-03-01 quarters 0.784343    4172      108            65              62         317


25000 клиентов в submission_quarters.csv


### Подготовка матрицы и проверка прогнозов

Сохранённые пять январских фолдов содержат прогнозы для клиентов, исключённых из обучения всех базовых моделей, включая отбор признаков и ранжировщик. Февраль и март используют обучение на предыдущих месяцах. Для полного пересчёта прогнозов задайте `rebuild = True` и используйте новый каталог кэша. При изменении данных, признаков или гиперпараметров также выберите новый каталог.

In [2]:
from pathlib import Path
from research.tail_inputs import prepare
from research.run_tail import run
from research.tail_review import review
import shutil, json

experiment_cache = Path('.checkpoints/tail6_notebook')
rebuild = False
prepare(globals(),experiment_cache,rebuild_scores=rebuild)

In [3]:
if not rebuild:
    for fold in range(5):
        filename = f'january_fold_{fold}.parquet'
        shutil.copyfile(Path('research/oof') / filename,experiment_cache / filename)
prediction, selected_model = run(experiment_cache)
february_choice = json.loads(Path('research/results/tail_february_choice.json').read_text())['model']
review(experiment_cache,february_choice)
print('Модель для апреля',selected_model)
display(pd.read_csv('research/results/tail_models.csv').query('model == @february_choice'))

2 lightgbm_plain_half  month               model     ap20  buyers  changed  added_buyers  dropped_buyers  false_7_15
     2 lightgbm_plain_half 0.778523    4170      151            89              85         336


3 lightgbm_plain_half  month               model     ap20  buyers  changed  added_buyers  dropped_buyers  false_7_15
     2 lightgbm_plain_half 0.778523    4170      151            89              85         336
     3 lightgbm_plain_half 0.785118    4182      124            83              73         322


Модель для апреля lightgbm_plain_half


,month,model,ap20,buyers,changed,added_buyers,dropped_buyers,false_7_15
14,2,lightgbm_plain_half,0.778523,4170,151,89,85,336
45,3,lightgbm_plain_half,0.785118,4182,124,83,73,322


### Проверенный прогноз в порядке образца Kaggle

In [4]:
submission_tail = write_submission(prediction,'submission_tail')
assert len(submission_tail) == len(test) == 25000
print('Строк',len(submission_tail),'· CSV и ZIP проверены')

Строк 25000 · CSV и ZIP проверены
